# 面试问题：Ring Attention / Context Parallel 怎样在多卡上保持精确 attention？

**回答主线。** Context Parallel 把序列轴切到多张设备；每张设备保留自己的 query block，并让 K/V block 沿 ring 依次流过。每收到一块，就用 online softmax 更新运行最大值、归一化分母和加权值，因此无需保存完整 `QK^T`，遍历全部块后仍与全量 attention 数值等价。它改变的是计算与通信组织，不是把二次 attention 近似成线性。

工程难点包括 causal 绝对位置、块序、通信—计算重叠、非整除分片、层/版本绑定和故障恢复。下面用 NumPy 模拟多个 rank，不调用分布式高层接口。


In [ ]:
import hashlib, math
from dataclasses import dataclass
import numpy as np

# 小张量足以验证 ring 顺序、causal mask 与全量 oracle。
rng152 = np.random.default_rng(152)

def stable_softmax152(scores, mask=None):
    scores = np.asarray(scores, dtype=np.float64)
    if mask is not None:
        scores = np.where(mask, scores, -np.inf)
    maximum = np.max(scores, axis=-1, keepdims=True)
    exp = np.where(np.isfinite(scores), np.exp(scores - maximum), 0.0)
    return exp / np.sum(exp, axis=-1, keepdims=True)

sanity152 = stable_softmax152(np.array([[1.0, 2.0]]))
assert sanity152.shape == (1, 2)
assert np.allclose(sanity152.sum(axis=1), 1.0)
assert sanity152[0, 1] > sanity152[0, 0]


## 1. 分片契约必须保留全局 token 位置

Rank 上的局部数组下标不能替代全局位置，否则 causal mask 和 RoPE 都会重置。非整除长度应显式分配余数；生产实现还要为 padding 与负载均衡选择策略。


In [ ]:
def shard_positions152(length, world_size):
    # 前 remainder 个 rank 多持有一个 token，且所有区间连续无重叠。
    base, remainder = divmod(length, world_size)
    shards, start = [], 0
    for rank in range(world_size):
        size = base + int(rank < remainder)
        shards.append(np.arange(start, start + size, dtype=np.int64))
        start += size
    return shards

shards152 = shard_positions152(10, 3)
assert [len(x) for x in shards152] == [4, 3, 3]
assert np.array_equal(np.concatenate(shards152), np.arange(10))
assert len(set(np.concatenate(shards152).tolist())) == 10


## 2. Online softmax 合并每个 KV block 的局部统计量

对每行 query 保存 `m`、`l` 和未归一化累积向量。新块到达时先抬高运行最大值，再按 `exp(old_m-new_m)` 重标旧贡献。某个 causal 块整行不可见时必须保持原状态，不能让 `-inf - -inf` 产生 NaN。


In [ ]:
def online_update152(q, k, v, state, valid):
    # 只更新至少有一个合法 key 的 query 行，空块保持原状态。
    m, l, acc = (x.copy() for x in state)
    scores = (q @ k.T) / math.sqrt(q.shape[1])
    rows = np.flatnonzero(np.any(valid, axis=1))
    if len(rows):
        masked = np.where(valid[rows], scores[rows], -np.inf)
        block_m = np.max(masked, axis=1)
        new_m = np.maximum(m[rows], block_m)
        old_scale = np.where(np.isfinite(m[rows]), np.exp(m[rows] - new_m), 0.0)
        weights = np.where(valid[rows], np.exp(masked - new_m[:, None]), 0.0)
        l[rows] = l[rows] * old_scale + weights.sum(axis=1)
        acc[rows] = acc[rows] * old_scale[:, None] + weights @ v
        m[rows] = new_m
    return m, l, acc

q2_152 = np.array([[1.0, 0.0], [0.0, 1.0]])
k2_152 = np.eye(2)
v2_152 = np.array([[2.0, 0.0], [0.0, 3.0]])
init152 = (np.full(2, -np.inf), np.zeros(2), np.zeros((2, 2)))
state152 = online_update152(q2_152, k2_152, v2_152, init152, np.ones((2, 2), bool))
assert np.all(state152[1] > 0)
assert np.isfinite(state152[2]).all()
assert state152[2].shape == (2, 2)


## 3. Blockwise online 结果应与全量 attention 等价

正确性 oracle 不是“看起来相似”，而是同一 Q/K/V 与 mask 下逐元素对比。块大小只影响访问顺序和舍入路径，不应改变数学定义。


In [ ]:
def full_attention152(q, k, v, valid):
    scores = (q @ k.T) / math.sqrt(q.shape[1])
    return stable_softmax152(scores, valid) @ v

def block_attention152(q, k, v, block_size, valid):
    # 按 K/V 块推进同一份在线状态，最后统一除以累计分母。
    state = (np.full(len(q), -np.inf), np.zeros(len(q)), np.zeros((len(q), v.shape[1])))
    for start in range(0, len(k), block_size):
        stop = min(start + block_size, len(k))
        state = online_update152(q, k[start:stop], v[start:stop], state, valid[:, start:stop])
    return state[2] / state[1][:, None]

q152 = rng152.normal(size=(7, 4)); k152 = rng152.normal(size=(9, 4)); v152 = rng152.normal(size=(9, 5))
all_valid152 = np.ones((7, 9), dtype=bool)
full152 = full_attention152(q152, k152, v152, all_valid152)
blocked152 = block_attention152(q152, k152, v152, 3, all_valid152)
assert blocked152.shape == full152.shape
assert np.allclose(blocked152, full152, atol=1e-12)
assert np.isfinite(blocked152).all()


## 4. Causal mask 使用 query/key 的全局绝对位置

每个 query 只能读 `key_pos <= query_pos`。局部块即使从零编号，也不能据此决定可见性。这个合同同样约束 RoPE、ALiBi 与滑窗策略。


In [ ]:
def causal_mask152(query_positions, key_positions):
    # 比较全局位置，而不是 rank 内数组下标。
    return key_positions[None, :] <= query_positions[:, None]

positions152 = np.arange(9)
causal152 = causal_mask152(positions152, positions152)
causal_full152 = full_attention152(k152, k152, v152, causal152)
causal_block152 = block_attention152(k152, k152, v152, 2, causal152)
assert np.array_equal(causal152, np.tril(np.ones((9, 9), bool)))
assert np.allclose(causal_full152, causal_block152, atol=1e-12)
assert causal152[-1].all() and causal152[0].sum() == 1


## 5. Ring schedule 让每个 rank 恰好看到所有 KV owner

第 `step` 轮每个 rank 处理当前持有的块，并把块发给邻居。无论采用顺时针还是逆时针，完整一圈必须无重复无遗漏；真实实现还会把通信放进独立 stream 与计算重叠。


In [ ]:
def ring_owners152(world_size, query_rank):
    # 这里模拟 KV block 每轮从前一个 rank 流入。
    return [(query_rank - step) % world_size for step in range(world_size)]

schedules152 = [ring_owners152(4, rank) for rank in range(4)]
assert all(len(set(row)) == 4 for row in schedules152)
assert all(set(row) == set(range(4)) for row in schedules152)
assert schedules152[0] == [0, 3, 2, 1]


## 6. 模拟 Context Parallel：Q 留在本地，KV 绕 ring

每个 rank 只为本地 query 保存输出状态，但依次读取全部 KV shard。最终按全局 query 位置拼接。这里故意使用非整除长度，验证分片边界不会改变结果。


In [ ]:
def ring_attention152(q, k, v, world_size, causal=True):
    # NumPy 模拟通信顺序；生产版本会在设备间发送 K/V buffer。
    shards = shard_positions152(len(q), world_size)
    output = np.empty((len(q), v.shape[1]), dtype=np.float64)
    for rank, qpos in enumerate(shards):
        state = (np.full(len(qpos), -np.inf), np.zeros(len(qpos)), np.zeros((len(qpos), v.shape[1])))
        for owner in ring_owners152(world_size, rank):
            kpos = shards[owner]
            valid = causal_mask152(qpos, kpos) if causal else np.ones((len(qpos), len(kpos)), bool)
            state = online_update152(q[qpos], k[kpos], v[kpos], state, valid)
        output[qpos] = state[2] / state[1][:, None]
    return output

ring152 = ring_attention152(k152, k152, v152, world_size=4, causal=True)
assert ring152.shape == causal_full152.shape
assert np.allclose(ring152, causal_full152, atol=1e-12)
assert np.isfinite(ring152).all()


## 7. 是否扩展取决于通信能否被块计算覆盖

每轮时间近似为 `max(compute, communication)`，而不是两者简单相加，前提是依赖与 stream 真能重叠。块太小会被启动开销支配，块太大则增加峰值内存和流水尾部。


In [ ]:
def ring_cost152(q_tokens, kv_tokens, dim, value_dim, bytes_per_elem, flops_per_s, bandwidth_bytes_s):
    # attention score 与加权 V 各计一次乘加，通信发送 K 和 V。
    flops = 2.0 * q_tokens * kv_tokens * (dim + value_dim)
    communicated = kv_tokens * (dim + value_dim) * bytes_per_elem
    compute_s = flops / flops_per_s
    comm_s = communicated / bandwidth_bytes_s
    return {"compute_s": compute_s, "comm_s": comm_s, "overlapped_s": max(compute_s, comm_s)}

cost152 = ring_cost152(1024, 1024, 128, 128, 2, 100e12, 100e9)
assert cost152["compute_s"] > 0 and cost152["comm_s"] > 0
assert cost152["overlapped_s"] == max(cost152["compute_s"], cost152["comm_s"])
assert cost152["overlapped_s"] <= cost152["compute_s"] + cost152["comm_s"]


## 8. Ring packet 需要层、步、owner 与内容完整性

一次 silent stale block 会污染整层输出。接收端应校验模型版本、层号、microbatch、ring step、owner 和 digest；超时后通常整轮失败重放，而不是把缺块当作零。


In [ ]:
@dataclass(frozen=True)
class RingPacket152:
    model_revision: str
    layer: int
    step: int
    owner: int
    digest: str

def make_packet152(array, revision, layer, step, owner):
    # 摘要绑定 dtype、shape 与原始字节，避免同字节不同布局误读。
    payload = str(array.dtype).encode() + repr(array.shape).encode() + array.tobytes()
    return RingPacket152(revision, layer, step, owner, hashlib.sha256(payload).hexdigest())

packet152 = make_packet152(k152[shards152[0]], "model-v3", 7, 0, 0)
same152 = make_packet152(k152[shards152[0]], "model-v3", 7, 0, 0)
changed152 = make_packet152(k152[shards152[0]] + 1e-3, "model-v3", 7, 0, 0)
assert packet152 == same152
assert packet152.digest != changed152.digest
assert packet152.layer == 7 and packet152.model_revision == "model-v3"


## 面试总结

- Context Parallel 切的是序列轴；每个 rank 保留 Q shard，K/V block 在设备间轮转。
- Online softmax 用运行最大值重标旧贡献，因此遍历所有合法块后仍是精确 attention。
- causal、RoPE 和窗口策略都必须基于全局位置；局部下标重置是高频错误。
- 性能取决于块计算能否覆盖通信、启动开销与拓扑，正确性则由全量 oracle 和 packet 合同保证。

延伸阅读：[Ring Attention](https://arxiv.org/abs/2310.01889)、[DeepSpeed-Ulysses](https://arxiv.org/abs/2309.14509)、[Context Parallelism for Million-Token Inference](https://arxiv.org/abs/2411.01783)。
